# 1. Instalação e Imports

In [ ]:
!pip install dbfread pyarrow unidecode pyreaddbc

from ftplib import FTP
from pathlib import Path
import zipfile
import shutil
import gc

import pandas as pd
from dbfread import DBF
import pyreaddbc



# 2. Criação das Pastas

In [2]:
BASE_DIR = Path("/content")

RAW_DIR = BASE_DIR / "dados_brutos"
AUX_DIR = BASE_DIR / "tabelas_auxiliares"
OUT_DIR = BASE_DIR / "dados_processados"

RAW_DIR.mkdir(parents=True, exist_ok=True)
AUX_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 3. Funções Auxiliares

## 3.1 Função para baixar arquivos do FTP

In [3]:
def baixar_arquivo_ftp(ftp_host, ftp_dir, nome_arquivo, destino_dir, forcar=False):
    destino_dir = Path(destino_dir)
    destino_dir.mkdir(parents=True, exist_ok=True)

    local_path = destino_dir / nome_arquivo

    if local_path.exists() and not forcar:
        print(f"Já existe: {local_path.name}, pulando.")
        return local_path

    if local_path.exists() and forcar:
        local_path.unlink()
        print(f"Arquivo antigo removido: {local_path.name}")

    ftp = FTP(ftp_host)
    ftp.login()

    try:
        ftp.cwd(ftp_dir)
    except Exception as e:
        ftp.quit()
        raise ValueError(f"Erro ao acessar diretório FTP '{ftp_dir}': {e}")

    try:
        print(f"Baixando {nome_arquivo}...")
        with open(local_path, "wb") as f:
            ftp.retrbinary(f"RETR {nome_arquivo}", f.write)

        tamanho = local_path.stat().st_size
        print(f"Salvo em: {local_path}")
        print(f"Tamanho: {tamanho} bytes")

        if tamanho == 0:
            raise ValueError("Arquivo baixado com tamanho zero.")

    except Exception as e:
        print(f"Erro ao baixar {nome_arquivo}: {e}")
        if local_path.exists():
            local_path.unlink()
            print("Arquivo corrompido removido.")
    finally:
        ftp.quit()

    return local_path

## 3.2 Função para ler DBF

In [4]:
def load_dbf(path):
    tabela = DBF(str(path), encoding="latin1", ignore_missing_memofile=True)
    return pd.DataFrame(iter(tabela))

## 3.3 Função para limpar códigos

In [5]:
def clean_code(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.lstrip("0")
    )

## 3.4 Função para inspecionar tabelas auxiliares

In [6]:
def inspecionar_tabela(nome_arquivo):
    df = load_dbf(AUX_DIR / nome_arquivo)
    print(f"\n=== {nome_arquivo} ===")
    print("Colunas:")
    print(df.columns.tolist())
    print("\nPrimeiras linhas:")
    display(df.head())
    return df

## 3.5 Função para procurar colunas por nome

In [7]:
def procurar_colunas(df, palavras):
    cols = []
    for c in df.columns:
        nome = c.upper()
        if any(p.upper() in nome for p in palavras):
            cols.append(c)
    return cols

## 3.6 Função de merge seguro

In [8]:
def merge_seguro(df_base, df_aux, left_on, right_on, cols_aux, how="left"):
    base = df_base.copy()
    aux = df_aux.copy()

    base[left_on] = clean_code(base[left_on])
    aux[right_on] = clean_code(aux[right_on])

    cols_usar = [right_on] + [c for c in cols_aux if c != right_on and c in aux.columns]
    aux = aux[cols_usar].drop_duplicates(subset=[right_on])

    return base.merge(aux, left_on=left_on, right_on=right_on, how=how)

# 4. Download dos arquivos principais do SINAN

In [ ]:
anos = list(range(2014, 2025))  # 2014 a 2024

ftp_host = "ftp.datasus.gov.br"
ftp_dir_viol = "dissemin/publicos/SINAN/DADOS/FINAIS"

for ano in anos:
    sufixo = str(ano)[-2:]
    nome_arquivo = f"VIOLBR{sufixo}.DBC"
    baixar_arquivo_ftp(ftp_host, ftp_dir_viol, nome_arquivo, RAW_DIR)

# 5. Download do ZIP das tabelas auxiliares

In [ ]:
ftp_dir_aux = "dissemin/publicos/SINAN/AUXILIAR"
nome_zip_aux = "TAB_SINANNET.zip"

zip_aux_path = baixar_arquivo_ftp(ftp_host, ftp_dir_aux, nome_zip_aux, AUX_DIR)

# 6. Extração do ZIP das tabelas auxiliares

In [ ]:
if zip_aux_path.exists():
    with zipfile.ZipFile(zip_aux_path, "r") as z:
        z.extractall(AUX_DIR)
    print("Arquivos auxiliares extraídos com sucesso.")
else:
    print("ZIP auxiliar não encontrado para extração.")

# 7. Conversão dos arquivos .DBC para .DBF

In [ ]:
dbc_files = sorted(RAW_DIR.glob("*.DBC"))
[a.name for a in dbc_files]

In [ ]:
for dbc_path in sorted(RAW_DIR.glob("*.DBC")):
    dbf_path = dbc_path.with_suffix(".DBF")

    if dbf_path.exists():
        print(f"Já existe: {dbf_path.name}, pulando.")
        continue

    try:
        print(f"Convertendo {dbc_path.name} -> {dbf_path.name}")
        pyreaddbc.dbc2dbf(str(dbc_path), str(dbf_path))
        print("OK")
    except Exception as e:
        print(f"Erro ao converter {dbc_path.name}: {e}")

# 8. Conferência dos arquivos disponíveis

In [14]:
#arquivos principais

dbf_files = sorted(RAW_DIR.glob("*.DBF"))
[a.name for a in dbf_files]

# arquivos auxiliares

aux_dbfs = sorted(AUX_DIR.glob("*.DBF"))
[a.name for a in aux_dbfs]

[]

# 9. Função para converter idade do SINAN

In [15]:
def idade_em_anos(cod):
    if pd.isna(cod):
        return None

    try:
        cod = int(cod)
    except:
        return None

    unidade = cod // 1000
    valor = cod % 1000

    if unidade == 1:   # horas
        return valor / 8760
    elif unidade == 2: # dias
        return valor / 365
    elif unidade == 3: # meses
        return valor / 12
    elif unidade == 4: # anos
        return valor
    else:
        return None

# 10. Função principal para processar os arquivos de violência

In [ ]:
# Essa é a função que filtra:
#   - mulheres
#   - Bahia
#   - Salvador
#   - exclui violência autoprovocada


def processar_dbf(path_dbf):
    nome = path_dbf.name

    try:
        ano = 2000 + int(nome[-6:-4])
    except:
        print(f"[AVISO] Não consegui extrair o ano de {nome}")
        return None

    print(f"\n=== Processando {nome} (ano {ano}) ===")

    tabela = DBF(str(path_dbf), encoding="latin1", ignore_missing_memofile=True)

    registros = []
    total = mantidos = 0

    for rec in tabela:
        total += 1

        sexo = str(rec.get("CS_SEXO") or "").strip()

        uf_raw = rec.get("SG_UF_NOT")
        uf_str = str(uf_raw).strip()

        mun_raw = rec.get("ID_MUNICIP")
        mun_str = str(mun_raw or "").strip().replace(".0", "").lstrip("0")

        les_aut = str(rec.get("LES_AUTOP") or "").strip()

        if sexo != "F":
            continue

        uf_ok = (uf_str == "29") or (uf_str.upper() == "BA")
        if not uf_ok:
            continue

        if not mun_str.startswith("292740"):
            continue

        if les_aut == "1":
            continue

        linha = dict(rec)
        linha["ano"] = ano
        registros.append(linha)

        mantidos += 1

    print(f"Total lidos: {total}")
    print(f"Registros mantidos: {mantidos}")

    if not registros:
        return None

    df = pd.DataFrame(registros)
    return df

# 11. Processar todos os anos

In [ ]:
dbf_files = sorted([p for p in RAW_DIR.iterdir() if p.suffix.upper() == ".DBF"])
dbf_files

In [ ]:
dfs = []

for arq in dbf_files:
    df_ano = processar_dbf(arq)
    if df_ano is not None and not df_ano.empty:
        dfs.append(df_ano)
        gc.collect()

sinan = pd.concat(dfs, ignore_index=True)

print(sinan.shape)
sinan.head()

# 12. Criar variáveis derivadas

## 12.1 Idade em anos

In [19]:
sinan["IDADE_ANOS"] = sinan["NU_IDADE_N"].apply(idade_em_anos)

## 12.2 Faixa etária

In [20]:
bins = [0, 9, 14, 19, 24, 34, 44, 59, 120]
labels = ["0-9", "10-14", "15-19", "20-24", "25-34", "35-44", "45-59", "60+"]

sinan["FAIXA_ETARIA"] = pd.cut(sinan["IDADE_ANOS"], bins=bins, labels=labels)

In [21]:
def agrupar_faixa_etaria(fx):
    if pd.isna(fx):
        return None

    fx = str(fx)

    if fx in ["0-9"]:
        return "Criança"

    if fx in ["10-14", "15-19"]:
        return "Adolescente"

    if fx in ["20-24"]:
        return "Jovem"

    if fx in ["25-34", "35-44", "45-59"]:
        return "Adulta"

    if fx in ["60+"]:
        return "Idosa"

    return None

sinan["FAIXA_ETARIA_AMPLA"] = sinan["FAIXA_ETARIA"].apply(agrupar_faixa_etaria)


# 13. Recodificação de variáveis

## 13.1 Função auxiliar

In [22]:
def aplicar_mapa(df, col_cod, col_desc, mapa):
    if col_cod not in df.columns:
        print(f"[AVISO] Coluna {col_cod} não encontrada.")
        return df

    df[col_cod] = df[col_cod].astype(str).str.strip()
    df[col_desc] = df[col_cod].map(mapa)
    return df

## 13.2 Mapas

In [23]:
map_sexo = {
    "M": "Masculino",
    "F": "Feminino",
    "I": "Ignorado"
}

map_raca = {
    "1": "Branca",
    "2": "Preta",
    "3": "Amarela",
    "4": "Parda",
    "5": "Indígena",
    "9": "Ignorado"
}

map_viol = {
    "1": "Sim",
    "2": "Não",
    "8": "Não se aplica",
    "9": "Ignorado"
}

map_escolaridade = {
    "1": "1ª a 4ª série incompleta do EF",
    "2": "4ª série completa do EF",
    "3": "5ª a 8ª série incompleta do EF",
    "4": "Ensino fundamental completo",
    "5": "Ensino médio incompleto",
    "6": "Ensino médio completo",
    "7": "Educação superior incompleta",
    "8": "Educação superior completa",
    "9": "Ignorado",
    "10": "Não se aplica"
}

map_sit_conjug = {
    "1": "Solteiro(a)",
    "2": "Casado(a)/União consensual",
    "3": "Viúvo(a)",
    "4": "Separado(a)",
    "8": "Não se aplica",
    "9": "Ignorado"
}

map_local_ocor = {
    "1": "Residência",
    "01": "Residência",
    "2": "Habitação coletiva",
    "02": "Habitação coletiva",
    "3": "Escola",
    "03": "Escola",
    "4": "Local de prática esportiva",
    "04": "Local de prática esportiva",
    "5": "Bar ou similar",
    "05": "Bar ou similar",
    "6": "Via pública",
    "06": "Via pública",
    "7": "Comércio/Serviços",
    "07": "Comércio/Serviços",
    "8": "Indústrias/Construção",
    "08": "Indústrias/Construção",
    "9": "Outro",
    "09": "Outro",
    "99": "Ignorado"
}

map_out_vezes = {
    "1": "Sim",
    "2": "Não",
    "9": "Ignorado"
}

map_gestante = {
    "1": "1º trimestre",
    "2": "2º trimestre",
    "3": "3º trimestre",
    "4": "Idade gestacional ignorada",
    "5": "Não",
    "6": "Não se aplica",
    "9": "Ignorado"
}

map_sim_nao = {
    "1": "Sim",
    "2": "Não",
    "8": "Não se aplica",
    "9": "Ignorado"
}

map_num_envolv = {
    "1": "Um",
    "2": "Dois ou mais",
    "9": "Ignorado"
}

map_lesao_nat = {
    "01": "Contusão",
    "02": "Corte/Laceração",
    "03": "Fratura",
    "04": "Entorse/Luxação",
    "05": "Traumatismo dentário",
    "06": "Traumatismo crânioencefálico",
    "07": "Politraumatismo",
    "08": "Envenenamento/Intoxicação",
    "09": "Queimadura",
    "10": "Outros",
    "11": "Sem lesão física",
    "88": "Não se aplica",
    "99": "Ignorado"
}

map_classi_fin = {
    "1": "Confirmado",
    "2": "Descartado",
    "3": "Inconclusivo/Provável",
    "8": "Ignorado"
}

map_evolucao = {
    "1": "Alta",
    "2": "Evasão/Fuga",
    "3": "Óbito por violência",
    "9": "Ignorado"
}


In [24]:
cols_viol = [
    "VIOL_FISIC", "VIOL_PSICO", "VIOL_TORT", "VIOL_SEXU",
    "VIOL_TRAF", "VIOL_FINAN", "VIOL_NEGLI", "VIOL_INFAN",
    "VIOL_LEGAL", "VIOL_OUTR", "VIOL_ESPEC"
]

for col in cols_viol:
    if col in sinan.columns:
        sinan = aplicar_mapa(sinan, col, col + "_DESC", map_viol)

In [25]:
def consolidar_tipos(row):
    tipos = []
    if row.get("VIOL_FISIC_DESC") == "Sim": tipos.append("Física")
    if row.get("VIOL_PSICO_DESC") == "Sim": tipos.append("Psicológica/Moral")
    if row.get("VIOL_TORT_DESC")  == "Sim": tipos.append("Tortura")
    if row.get("VIOL_SEXU_DESC")  == "Sim": tipos.append("Sexual")
    if row.get("VIOL_TRAF_DESC")  == "Sim": tipos.append("Tráfico de pessoas")
    if row.get("VIOL_FINAN_DESC") == "Sim": tipos.append("Financeira/Econômica")
    if row.get("VIOL_NEGLI_DESC") == "Sim": tipos.append("Negligência/Abandono")
    if row.get("VIOL_INFAN_DESC") == "Sim": tipos.append("Trabalho infantil")
    if row.get("VIOL_LEGAL_DESC") == "Sim": tipos.append("Intervenção legal")
    if row.get("VIOL_OUTR_DESC")  == "Sim": tipos.append("Outras")
    if row.get("VIOL_ESPEC_DESC") == "Sim": tipos.append("Situação específica")
    return ", ".join(tipos) if tipos else None

sinan["TIPO_VIOLENCIA"] = sinan.apply(consolidar_tipos, axis=1)

## 13.3 Aplicar recodificação

In [26]:
sinan = aplicar_mapa(sinan, "CS_SEXO", "CS_SEXO_DESC", map_sexo)
sinan = aplicar_mapa(sinan, "CS_RACA", "CS_RACA_DESC", map_raca)
sinan = aplicar_mapa(sinan, "CS_ESCOL_N", "ESCOLARIDADE_DESC", map_escolaridade)
sinan = aplicar_mapa(sinan, "SIT_CONJUG", "SITUACAO_CONJUGAL_DESC", map_sit_conjug)
sinan = aplicar_mapa(sinan, "LOCAL_OCOR", "LOCAL_OCOR_DESC", map_local_ocor)
sinan = aplicar_mapa(sinan, "OUT_VEZES", "OUT_VEZES_DESC", map_out_vezes)
sinan = aplicar_mapa(sinan, "CS_GESTANT", "GESTANTE_DESC", map_gestante)
sinan = aplicar_mapa(sinan, "AUTOR_ALCO", "AUTOR_ALCO_DESC", map_sim_nao)
sinan = aplicar_mapa(sinan, "NUM_ENVOLV", "NUM_ENVOLV_DESC", map_num_envolv)
sinan = aplicar_mapa(sinan, "REL_TRAB", "REL_TRAB_DESC", map_sim_nao)
sinan = aplicar_mapa(sinan, "LESAO_NAT", "LESAO_NAT_DESC", map_lesao_nat)
sinan["TIPO_VIOLENCIA"] = sinan.apply(consolidar_tipos, axis=1)
sinan = aplicar_mapa(sinan, "CLASSI_FIN", "CLASSI_FIN_DESC", map_classi_fin)
sinan = aplicar_mapa(sinan, "EVOLUCAO", "EVOLUCAO_DESC", map_evolucao)

# 14. Organizando os arquivos auxiliares

In [ ]:
pasta_origem = AUX_DIR / "TAB_SINANNET"
pasta_destino = AUX_DIR

if not pasta_origem.exists():
    print("Pasta TAB_SINANNET não encontrada.")
else:
    arquivos_movidos = 0

    for arq in pasta_origem.glob("*.DBF"):
        destino = pasta_destino / arq.name

        if destino.exists():
            destino.unlink()

        shutil.move(str(arq), str(destino))
        arquivos_movidos += 1

    shutil.rmtree(pasta_origem)

    print(f"{arquivos_movidos} arquivos .DBF movidos para {pasta_destino}")
    print("Subpasta TAB_SINANNET removida completamente.")

# 15. Ler as tabelas auxiliares realmente úteis

* OCUPANET.DBF
* UNIDTOTAL.DBF
* UNIDANET.DBF
* UNIDINVEST.DBF
* DISTRNET.DBF

In [28]:
df_ocupa = load_dbf(AUX_DIR / "OCUPANET.DBF")
df_unidtotal = load_dbf(AUX_DIR / "UNIDTOTAL.DBF")
df_unidanet = load_dbf(AUX_DIR / "UNIDANET.DBF")
df_unidinvest = load_dbf(AUX_DIR / "UNIDINVEST.DBF")

# 16. Inspecionar as colunas antes dos *merges*

In [ ]:
display(df_ocupa.head())
display(df_unidtotal.head())
display(df_unidanet.head())
display(df_unidinvest.head())

# 17. Merge com OCUPANET

In [30]:
COD_OCUPA_AUX = "ID_OCUPA_N"
DESC_OCUPA_AUX = "NM_OCUPACA"

In [31]:
sinan = merge_seguro(
    sinan,
    df_ocupa,
    left_on="ID_OCUPA_N",
    right_on=COD_OCUPA_AUX,
    cols_aux=[COD_OCUPA_AUX, DESC_OCUPA_AUX]
)

sinan = sinan.rename(columns={DESC_OCUPA_AUX: "OCUPACAO_DESC"})

# 18. Merge com UNIDTOTAL

In [32]:
COD_UNIDADE_AUX = "ID_UNIDADE"
NOME_UNIDADE_AUX = "NM_UNIDADE"
BAIRRO_UNIDADE_AUX = "NM_BAIRRO"
MUNIC_UNIDADE_AUX = "ID_MUNICIP"

In [33]:
sinan = merge_seguro(
    sinan,
    df_unidtotal,
    left_on="ID_UNIDADE",
    right_on=COD_UNIDADE_AUX,
    cols_aux=[COD_UNIDADE_AUX, NOME_UNIDADE_AUX, BAIRRO_UNIDADE_AUX, MUNIC_UNIDADE_AUX]
)

In [34]:
rename_map = {}
if NOME_UNIDADE_AUX in sinan.columns:
    rename_map[NOME_UNIDADE_AUX] = "UNIDADE_DESC"
if BAIRRO_UNIDADE_AUX in sinan.columns:
    rename_map[BAIRRO_UNIDADE_AUX] = "BAIRRO_UNIDADE"
if MUNIC_UNIDADE_AUX in sinan.columns:
    rename_map[MUNIC_UNIDADE_AUX] = "MUNICIPIO_UNIDADE"

sinan = sinan.rename(columns=rename_map)

# 19. Fallback com UNIDANET

In [35]:
COD_UNIDANET = "ID_UNIDADE"
BAIRRO_UNIDANET = "NM_BAIRRO"

In [36]:
sinan_aux = merge_seguro(
    sinan[["ID_UNIDADE"]].copy(),
    df_unidanet,
    left_on="ID_UNIDADE",
    right_on=COD_UNIDANET,
    cols_aux=[COD_UNIDANET, BAIRRO_UNIDANET]
)

sinan["BAIRRO_UNIDADE"] = sinan["BAIRRO_UNIDADE"].fillna(sinan_aux[BAIRRO_UNIDANET])

# 20. Fallback com UNIDINVEST

In [37]:
COD_UNIDINVEST = "ID_UNIDADE"
BAIRRO_UNIDINVEST = "NM_BAIRRO"

In [38]:
sinan_aux2 = merge_seguro(
    sinan[["ID_UNIDADE"]].copy(),
    df_unidinvest,
    left_on="ID_UNIDADE",
    right_on=COD_UNIDINVEST,
    cols_aux=[COD_UNIDINVEST, BAIRRO_UNIDINVEST]
)

sinan["BAIRRO_UNIDADE"] = sinan["BAIRRO_UNIDADE"].fillna(sinan_aux2[BAIRRO_UNIDINVEST])

# 21. Criar coluna final de bairro

In [39]:
sinan["BAIRRO_ANALISE"] = sinan.get("BAIRRO_UNIDADE")
sinan["OBS_BAIRRO"] = "Bairro associado à unidade notificadora/investigadora"

# 22. Enriquecimento Análitico

## 22.1 Relação vítima-autor

In [40]:
cols_rel = [
    "REL_PAI", "REL_MAE", "REL_PAD", "REL_MAD", "REL_CONJ", "REL_EXCON",
    "REL_NAMO", "REL_EXNAM", "REL_FILHO", "REL_IRMAO",
    "REL_CONHEC", "REL_CUIDA", "REL_PATRAO", "REL_INST",
    "REL_POL", "REL_PROPRI", "REL_DESCO", "REL_OUTROS"
]

for col in cols_rel:
    if col in sinan.columns:
        sinan = aplicar_mapa(sinan, col, col + "_DESC", map_sim_nao)

In [41]:
def consolidar_relacao(row):
    relacoes = []

    mapa_rel = {
        "REL_PAI_DESC": "Pai",
        "REL_MAE_DESC": "Mãe",
        "REL_PAD_DESC": "Padrasto",
        "REL_MAD_DESC": "Madrasta",
        "REL_CONJ_DESC": "Cônjuge",
        "REL_EXCON_DESC": "Ex-cônjuge",
        "REL_NAMO_DESC": "Namorado(a)",
        "REL_EXNAM_DESC": "Ex-namorado(a)",
        "REL_FILHO_DESC": "Filho(a)",
        "REL_IRMAO_DESC": "Irmão(ã)",
        "REL_CONHEC_DESC": "Conhecido(a)",
        "REL_CUIDA_DESC": "Cuidador(a)",
        "REL_PATRAO_DESC": "Patrão/Chefe",
        "REL_INST_DESC": "Relação institucional",
        "REL_POL_DESC": "Policial/Agente da lei",
        "REL_PROPRI_DESC": "Própria pessoa",
        "REL_DESCO_DESC": "Desconhecido(a)",
        "REL_OUTROS_DESC": "Outros"
    }

    for col, nome in mapa_rel.items():
        if row.get(col) == "Sim":
            relacoes.append(nome)

    return ", ".join(relacoes) if relacoes else None

sinan["RELACAO_AUTOR"] = sinan.apply(consolidar_relacao, axis=1)

In [42]:
def agrupar_relacao_autor(rel):
    if pd.isna(rel):
        return None

    rel = str(rel)

    parceiro_intimo = [
        "Cônjuge",
        "Ex-cônjuge",
        "Namorado(a)",
        "Ex-namorado(a)"
    ]

    familiar = [
        "Pai",
        "Mãe",
        "Padrasto",
        "Madrasta",
        "Filho(a)",
        "Irmão(ã)"
    ]

    conhecido = [
        "Conhecido(a)",
        "Cuidador(a)",
        "Patrão/Chefe"
    ]

    institucional = [
        "Relação institucional",
        "Policial/Agente da lei"
    ]

    if any(x in rel for x in parceiro_intimo):
        return "Parceiro íntimo"

    if any(x in rel for x in familiar):
        return "Familiar"

    if any(x in rel for x in conhecido):
        return "Conhecido"

    if "Desconhecido(a)" in rel:
        return "Desconhecido"

    if any(x in rel for x in institucional):
        return "Institucional"

    return "Outros"

In [43]:
sinan["RELACAO_AUTOR_GRUPO"] = sinan["RELACAO_AUTOR"].apply(agrupar_relacao_autor)

## 22.2 Tipo de Agressão

In [44]:
cols_agressao = [
    "AG_FORCA", "AG_ENFOR", "AG_OBJETO", "AG_CORTE",
    "AG_QUENTE", "AG_ENVEN", "AG_FOGO", "AG_AMEACA",
    "AG_OUTROS"
]

for col in cols_agressao:
    if col in sinan.columns:
        sinan = aplicar_mapa(sinan, col, col + "_DESC", map_sim_nao)

In [45]:
def consolidar_agressao(row):
    tipos = []

    mapa_ag = {
        "AG_FORCA_DESC": "Força corporal/espancamento",
        "AG_ENFOR_DESC": "Enforcamento",
        "AG_OBJETO_DESC": "Objeto contundente",
        "AG_CORTE_DESC": "Objeto perfurocortante",
        "AG_QUENTE_DESC": "Substância/objeto quente",
        "AG_ENVEN_DESC": "Envenenamento",
        "AG_FOGO_DESC": "Arma de fogo",
        "AG_AMEACA_DESC": "Ameaça",
        "AG_OUTROS_DESC": "Outros"
    }

    for col, nome in mapa_ag.items():
        if row.get(col) == "Sim":
            tipos.append(nome)

    return ", ".join(tipos) if tipos else None

sinan["TIPO_AGRESSAO"] = sinan.apply(consolidar_agressao, axis=1)

In [46]:
def agrupar_tipo_agressao(tipo):
    if pd.isna(tipo):
        return None

    tipo = str(tipo)

    if "Força corporal/espancamento" in tipo or "Enforcamento" in tipo:
        return "Força física"

    if "Objeto perfurocortante" in tipo:
        return "Arma branca"

    if "Arma de fogo" in tipo:
        return "Arma de fogo"

    if "Ameaça" in tipo:
        return "Ameaça"

    if "Objeto contundente" in tipo:
        return "Objeto contundente"

    if "Envenenamento" in tipo or "Substância/objeto quente" in tipo:
        return "Meio químico/térmico"

    return "Outros meios"

In [47]:
sinan["TIPO_AGRESSAO_GRUPO"] = sinan["TIPO_AGRESSAO"].apply(agrupar_tipo_agressao)

## 22.3 Lesão corporal

In [48]:
def grupo_cid(cod):
    if pd.isna(cod):
        return None

    cod = str(cod).strip().upper()

    if cod == "":
        return None

    if cod.startswith(("S", "T")):
        return "Lesões e traumatismos"

    if cod.startswith(("X", "Y")):
        return "Agressões e causas externas"

    if cod.startswith("R"):
        return "Sintomas e sinais"

    return "Outros CID"

sinan = aplicar_mapa(sinan, "LESAO_CORP", "LESAO_CORP_DESC", map_lesao_nat)
sinan["CIRC_LESAO_GRUPO"] = sinan["CIRC_LESAO"].apply(grupo_cid)

In [49]:
def resumo_lesao(row):
    partes = []

    if pd.notna(row.get("LESAO_NAT_DESC")):
        partes.append(str(row.get("LESAO_NAT_DESC")))

    if pd.notna(row.get("LESAO_CORP_DESC")):
        partes.append(f"Lesão corporal: {row.get('LESAO_CORP_DESC')}")

    if pd.notna(row.get("CIRC_LESAO_GRUPO")):
        partes.append(f"CID: {row.get('CIRC_LESAO_GRUPO')}")

    return " | ".join(partes) if partes else None

sinan["RESUMO_LESAO"] = sinan.apply(resumo_lesao, axis=1)

## 22.4 Encaminhamento da notificação

In [50]:
cols_enc = [
    "ENC_SAUDE", "ENC_TUTELA", "ENC_VARA", "ENC_ABRIGO",
    "ENC_SENTIN", "ENC_DEAM", "ENC_DPCA", "ENC_DELEG",
    "ENC_MPU", "ENC_MULHER", "ENC_CREAS", "ENC_IML",
    "ENC_OUTR"
]

for col in cols_enc:
    if col in sinan.columns:
        sinan = aplicar_mapa(sinan, col, col + "_DESC", map_sim_nao)

In [ ]:
def consolidar_encaminhamento(row):
    destinos = []

    mapa_enc = {
        "ENC_SAUDE_DESC": "Rede de saúde",
        "ENC_TUTELA_DESC": "Conselho tutelar",
        "ENC_VARA_DESC": "Vara/Justiça",
        "ENC_ABRIGO_DESC": "Abrigo",
        "ENC_SENTIN_DESC": "Unidade sentinela",
        "ENC_DEAM_DESC": "Delegacia da mulher",
        "ENC_DPCA_DESC": "Delegacia da criança/adolescente",
        "ENC_DELEG_DESC": "Delegacia comum",
        "ENC_MPU_DESC": "Ministério Público",
        "ENC_MULHER_DESC": "Rede de apoio à mulher",
        "ENC_CREAS_DESC": "CREAS",
        "ENC_IML_DESC": "IML",
        "ENC_OUTR_DESC": "Outros"
    }

    for col, nome in mapa_enc.items():
        if row.get(col) == "Sim":
            destinos.append(nome)

    return ", ".join(destinos) if destinos else None

sinan["ENCAMINHAMENTO_NOTIFICACAO"] = sinan.apply(consolidar_encaminhamento, axis=1)

## 22.5 Sexo do agressor



In [ ]:
map_autor_sexo = {
    "1": "Masculino",
    "2": "Feminino",
    "3": "Ambos",
    "9": "Ignorado"
}

sinan = aplicar_mapa(sinan, "AUTOR_SEXO", "AUTOR_SEXO_DESC", map_autor_sexo)

## 22.6 Rede de apoio acionada

In [ ]:
cols_rede = [
    "ATEND_MULH", "ASSIST_SOC", "REDE_SAU",
    "REDE_EDUCA", "DELEG_MULH", "DEFEN_PUBL"
]

for col in cols_rede:
    if col in sinan.columns:
        sinan = aplicar_mapa(sinan, col, col + "_DESC", map_sim_nao)

In [ ]:
def consolidar_rede(row):
    rede = []

    mapa_rede = {
        "ATEND_MULH_DESC": "Atendimento à mulher",
        "ASSIST_SOC_DESC": "Assistência social",
        "REDE_SAU_DESC": "Rede de saúde",
        "REDE_EDUCA_DESC": "Rede educacional",
        "DELEG_MULH_DESC": "Delegacia da mulher",
        "DEFEN_PUBL_DESC": "Defensoria pública"
    }

    for col, nome in mapa_rede.items():
        if row.get(col) == "Sim":
            rede.append(nome)

    return ", ".join(rede) if rede else None

sinan["REDE_APOIO_ACIONADA"] = sinan.apply(consolidar_rede, axis=1)

## 22.7 Deficiência

In [ ]:
cols_def = [
    "DEF_TRANS", "DEF_FISICA", "DEF_MENTAL",
    "DEF_VISUAL", "DEF_AUDITI", "TRAN_MENT",
    "TRAN_COMP", "DEF_OUT"
]

for col in cols_def:
    if col in sinan.columns:
        sinan = aplicar_mapa(sinan, col, col + "_DESC", map_sim_nao)

In [56]:
def consolidar_deficiencias(row):
    defs = []

    mapa_def = {
        "DEF_TRANS_DESC": "Deficiência intelectual",
        "DEF_FISICA_DESC": "Deficiência física",
        "DEF_MENTAL_DESC": "Transtorno mental",
        "DEF_VISUAL_DESC": "Deficiência visual",
        "DEF_AUDITI_DESC": "Deficiência auditiva",
        "TRAN_MENT_DESC": "Transtorno mental",
        "TRAN_COMP_DESC": "Transtorno comportamental",
        "DEF_OUT_DESC": "Outras deficiências"
    }

    for col, nome in mapa_def.items():
        if row.get(col) == "Sim":
            defs.append(nome)

    return ", ".join(defs) if defs else None

In [ ]:
sinan["DEFICIENCIAS_VITIMA"] = sinan.apply(consolidar_deficiencias, axis=1)
sinan["POSSUI_DEFICIENCIA"] = sinan["DEFICIENCIAS_VITIMA"].notna().map({True: "Sim", False: "Não"})

## 22.8 Desfecho do caso

In [58]:
def agrupar_evolucao(ev):
    if pd.isna(ev):
        return None

    ev = str(ev)

    if "Alta" in ev:
        return "Sem desfecho grave"

    if "Óbito" in ev:
        return "Óbito"

    if "Evasão" in ev:
        return "Interrupção do cuidado"

    return "Ignorado/Outros"

In [ ]:
sinan["EVOLUCAO_GRUPO"] = sinan["EVOLUCAO_DESC"].apply(agrupar_evolucao)

In [ ]:
print(sinan["CLASSI_FIN"].astype(str).value_counts(dropna=False).head(20))
print(sinan["EVOLUCAO"].astype(str).value_counts(dropna=False).head(20))

# 23. Datas e variáveis temporais

## 23.1 Datas

In [61]:
cols_data = ["DT_NOTIFIC", "DT_OCOR", "DT_INVEST", "DT_ENCERRA", "DT_OBITO"]

for col in cols_data:
    if col in sinan.columns:
        sinan[col] = pd.to_datetime(sinan[col], errors="coerce", dayfirst=True)

In [62]:
if "ano" in sinan.columns:
    sinan = sinan.rename(columns={"ano": "ANO_REFERENCIA_DADOS"})

In [63]:
if "DT_NOTIFIC" in sinan.columns:
    sinan["ANO_NOTIFIC"] = sinan["DT_NOTIFIC"].dt.year
    sinan["MES_NOTIFIC"] = sinan["DT_NOTIFIC"].dt.month
    sinan["ANO_MES_NOTIFIC"] = sinan["DT_NOTIFIC"].dt.to_period("M").astype(str)

if "DT_OCOR" in sinan.columns:
    sinan["ANO_OCOR"] = sinan["DT_OCOR"].dt.year
    sinan["MES_OCOR"] = sinan["DT_OCOR"].dt.month
    sinan["ANO_MES_OCOR"] = sinan["DT_OCOR"].dt.to_period("M").astype(str)

In [64]:
if "DT_NOTIFIC" in sinan.columns:
    sinan["TRIMESTRE_NOTIFIC"] = sinan["DT_NOTIFIC"].dt.quarter
    sinan["SEMESTRE_NOTIFIC"] = sinan["MES_NOTIFIC"].apply(lambda x: 1 if pd.notna(x) and x <= 6 else (2 if pd.notna(x) else None))
    sinan["MES_NOME_NOTIFIC"] = sinan["DT_NOTIFIC"].dt.month_name()

In [65]:
if "DT_OCOR" in sinan.columns:
    sinan["TRIMESTRE_OCOR"] = sinan["DT_OCOR"].dt.quarter
    sinan["SEMESTRE_OCOR"] = sinan["MES_OCOR"].apply(lambda x: 1 if pd.notna(x) and x <= 6 else (2 if pd.notna(x) else None))
    sinan["MES_NOME_OCOR"] = sinan["DT_OCOR"].dt.month_name()

## 23.2 Horas

In [66]:
def extrair_hora(h):
    if pd.isna(h):
        return None

    h = str(h).strip()

    if h == "":
        return None

    try:
        return int(h[:2])
    except:
        return None

In [67]:
sinan["HORA_OCOR_HORA"] = sinan["HORA_OCOR"].apply(extrair_hora)

In [68]:
def agrupar_turno(hora):
    if pd.isna(hora):
        return None

    hora = int(hora)

    if 0 <= hora < 6:
        return "Madrugada"
    elif 6 <= hora < 12:
        return "Manhã"
    elif 12 <= hora < 18:
        return "Tarde"
    else:
        return "Noite"

In [69]:
sinan["TURNO_OCORRENCIA"] = sinan["HORA_OCOR_HORA"].apply(agrupar_turno)

# 24. Base Analítica

In [70]:
cols_analiticas = [

    # =========================
    # 1. Metadados e vigilância
    # =========================
    "ANO_REFERENCIA_DADOS",
    "DT_NOTIFIC",
    "CLASSI_FIN_DESC",
    "EVOLUCAO_DESC",
    "EVOLUCAO_GRUPO",

    # =========================
    # 2. Temporalidade da notificação
    # =========================
    "ANO_NOTIFIC",
    "MES_NOTIFIC",
    "MES_NOME_NOTIFIC",
    "TRIMESTRE_NOTIFIC",
    "SEMESTRE_NOTIFIC",
    "ANO_MES_NOTIFIC",

    # =========================
    # 3. Temporalidade da ocorrência
    # =========================
    "DT_OCOR",
    "ANO_OCOR",
    "MES_OCOR",
    "MES_NOME_OCOR",
    "TRIMESTRE_OCOR",
    "SEMESTRE_OCOR",
    "ANO_MES_OCOR",

    # =========================
    # 4. Perfil da vítima
    # =========================
    "IDADE_ANOS",
    "FAIXA_ETARIA",
    "FAIXA_ETARIA_AMPLA",
    "CS_SEXO_DESC",
    "CS_RACA_DESC",
    "ESCOLARIDADE_DESC",
    "SITUACAO_CONJUGAL_DESC",
    "GESTANTE_DESC",
    "DEFICIENCIAS_VITIMA",
    "POSSUI_DEFICIENCIA",
    "OCUPACAO_DESC",

    # =========================
    # 5. Contexto da ocorrência
    # =========================
    "LOCAL_OCOR_DESC",
    "HORA_OCOR",
    "TURNO_OCORRENCIA",
    "OUT_VEZES_DESC",
    "REL_TRAB_DESC",

    # =========================
    # 6. Violência e dinâmica do evento
    # =========================
    "TIPO_VIOLENCIA",
    "TIPO_AGRESSAO",
    "TIPO_AGRESSAO_GRUPO",
    "NUM_ENVOLV_DESC",
    "AUTOR_ALCO_DESC",

    # =========================
    # 7. Perfil do provável autor
    # =========================
    "AUTOR_SEXO_DESC",
    "RELACAO_AUTOR",
    "RELACAO_AUTOR_GRUPO",

    # =========================
    # 8. Lesão e gravidade
    # =========================
    "LESAO_NAT_DESC",
    "LESAO_CORP_DESC",
    "CIRC_LESAO_GRUPO",
    "RESUMO_LESAO",

    # =========================
    # 9. Resposta institucional
    # =========================
    "ENCAMINHAMENTO_NOTIFICACAO",
    "REDE_APOIO_ACIONADA",
    "UNIDADE_DESC",

    # =========================
    # 10. Territorialização
    # =========================
    "BAIRRO_ANALISE",
    "OBS_BAIRRO"
]

cols_analiticas_existentes = [c for c in cols_analiticas if c in sinan.columns]
sinan_analitico = sinan[cols_analiticas_existentes].copy()

# 25. Conferências Finais

In [ ]:
print("Base completa:", sinan.shape)
print("Base analítica:", sinan_analitico.shape)

In [ ]:
cols_conf = [
    "ANO_REFERENCIA_DADOS",
    "CLASSI_FIN_DESC",
    "EVOLUCAO_DESC",
    "FAIXA_ETARIA",
    "FAIXA_ETARIA_AMPLA",
    "CS_RACA_DESC",
    "POSSUI_DEFICIENCIA",
    "LOCAL_OCOR_DESC",
    "TURNO_OCORRENCIA",
    "TIPO_VIOLENCIA",
    "TIPO_AGRESSAO_GRUPO",
    "RELACAO_AUTOR_GRUPO",
    "LESAO_NAT_DESC",
    "CIRC_LESAO_GRUPO",
    "ENCAMINHAMENTO_NOTIFICACAO",
    "REDE_APOIO_ACIONADA",
    "UNIDADE_DESC",
    "BAIRRO_ANALISE"
]

display(sinan[[c for c in cols_conf if c in sinan.columns]].head())

In [ ]:
cols_qualidade = [
    "TIPO_VIOLENCIA",
    "RELACAO_AUTOR",
    "TIPO_AGRESSAO",
    "LESAO_NAT_DESC",
    "ENCAMINHAMENTO_NOTIFICACAO",
    "REDE_APOIO_ACIONADA",
    "BAIRRO_ANALISE",
    "UNIDADE_DESC",
    "TURNO_OCORRENCIA",
    "POSSUI_DEFICIENCIA"
]

for col in cols_qualidade:
    if col in sinan.columns:
        pct = round(100 * sinan[col].isna().mean(), 2)
        print(f"{col}: {pct}% ausente")

# 26. Salvar a base completa e a base analítica

In [ ]:
out_parquet = OUT_DIR / "sinan_SSA_mulheres_2014_2024_final.parquet"
out_csv = OUT_DIR / "sinan_SSA_mulheres_2014_2024_final.csv"

sinan.to_parquet(out_parquet, index=False)
sinan.to_csv(out_csv, sep=";", index=False)

print(out_parquet)
print(out_csv)

In [ ]:
out_parquet_analitico = OUT_DIR / "sinan_SSA_mulheres_2014_2024_analitico.parquet"
out_csv_analitico = OUT_DIR / "sinan_SSA_mulheres_2014_2024_analitico.csv"

sinan_analitico.to_parquet(out_parquet_analitico, index=False)
sinan_analitico.to_csv(out_csv_analitico, sep=";", index=False)

print(out_parquet_analitico)
print(out_csv_analitico)